# libs

In [60]:
%pip install pandas nltk gensim pyLDAvis

import pandas as pd

import sklearn
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models import CoherenceModel
from nltk.stem import WordNetLemmatizer, SnowballStemmer
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package wordnet to /Users/Licas/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/Licas/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# dataset 

In [61]:
#import du csv
csv_file = pd.read_csv(r"banking77.csv", sep = ";")

# transformation en DF
df = pd.DataFrame(csv_file, index= None)
df

# on drop la colonne "Unnamed: 0" qui ne sert à rien"

df = df.drop(columns=['Unnamed: 0'], errors='ignore')
print(df)

                                                    text  label
0                         I am still waiting on my card?     11
1      What can I do if my card still hasn't arrived ...     11
2      I have been waiting over a week. Is the card s...     11
3      Can I track my card while it is in the process...     11
4      How do I know if I will get my card, or if it ...     11
...                                                  ...    ...
9998              You provide support in what countries?     24
9999                  What countries are you supporting?     24
10000                What countries are getting support?     24
10001                     Are cards available in the EU?     24
10002                   Which countries are represented?     24

[10003 rows x 2 columns]


In [62]:
df["text"].isna().sum()

np.int64(0)

In [63]:
df["label"].value_counts().sort_values(ascending= False)

label
15    187
28    182
6     181
75    180
19    177
     ... 
41     82
18     61
10     59
72     41
23     35
Name: count, Length: 77, dtype: int64

# on passe la colonne text dans une liste python

In [64]:
liste_textes = df["text"].dropna().tolist()
print(liste_textes[0])

I am still waiting on my card?


# il nous faut désormais nettoyer la liste de mots (stopwords + lemmatization)

In [65]:
stemmer = SnowballStemmer('english')

stopwords_list = {
    # Pronoms et articles
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", 
    'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'herself', 'it', "it's", 
    'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'the', 'a', 'an',
    
    # Prépositions et conjonctions
    'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 
    'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 
    'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 
    'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 
    'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 
    'other', 'some', 'such', 'only', 'own', 'same', 'so', 'than', 'too', 'very',
    
    # Verbes auxiliaires et d'état
    'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 
    'having', 'do', 'does', 'did', 'doing', 'can', 'could', 'should', 'would', 'will', 'just',
    
    # Stopwords métiers
    'card', 'money', 'account',
    
    #Verbes useless
    'get', 'need', 'want', 'like', 'know', 'tell', 'think',
    'look', 'work', 'try', 'help', 'come', 'happen', 'use', 'make'
}


stopwords_eng = STOPWORDS.union(stopwords_list)


def lemmatize_stemming(token):
    lemme= WordNetLemmatizer().lemmatize(token, pos='n')
    return stemmer.stem(lemme)

def preprocess(text):
    tokens = []
    for token in simple_preprocess(text, deacc=True):
        if token not in stopwords_eng and len(token) > 3:
            tokens.append(lemmatize_stemming(token))
    return tokens

# on applique nos fonctions à notre corpus pour les nettoyer

In [66]:
# on applique notre fonction à notre liste de textes pour obtenir "processed_docs"

processed_docs= [preprocess(doc) for doc in liste_textes]

# on récupère les categorie_id de notre df original + tokens
for titre, tokens in zip(df['label'], processed_docs):
    print(titre, '->', tokens)

11 -> ['wait']
11 -> ['hasn', 'arriv', 'week']
11 -> ['wait', 'week', 'come']
11 -> ['track', 'process', 'deliveri']
11 -> ['lost']
11 -> ['send']
11 -> ['info', 'deliveri']
11 -> ['receiv']
11 -> ['packag', 'track']
11 -> ['order']
11 -> []
11 -> ['haven', 'receiv', 'week', 'lost']
11 -> ['track']
11 -> ['track', 'deliveri']
11 -> ['week', 'order']
11 -> ['abl', 'track', 'sent']
11 -> ['week', 'worri']
11 -> ['go', 'arriv']
11 -> ['deliv', 'home', 'go']
11 -> ['week', 'issu', 'wait']
11 -> ['come', 'track', 'info']
11 -> ['status', 'deliveri']
11 -> ['sent']
11 -> []
11 -> []
11 -> ['week', 'sent']
11 -> ['track', 'sent']
11 -> ['reciev']
11 -> []
11 -> ['expect', 'deliveri', 'date']
11 -> ['track', 'number', 'sent']
11 -> ['track', 'sent']
11 -> ['lost', 'deliveri']
11 -> ['hasn', 'came']
11 -> ['order', 'week']
11 -> ['updat', 'replac']
11 -> ['long', 'ship']
11 -> ['track', 'sent']
11 -> ['haven', 'receiv', 'worri', 'lost']
11 -> ['hasn', 'deliv']
11 -> ['order']
11 -> ['credit', '

# on construit le dictionnaire Gensin pour obtenir une valeur numérique pour chaque mot



In [67]:
dictionary= gensim.corpora.Dictionary(processed_docs)

print(dictionary.token2id)
print('Nombre de mots gardés :', len(dictionary))

{'wait': 0, 'arriv': 1, 'hasn': 2, 'week': 3, 'come': 4, 'deliveri': 5, 'process': 6, 'track': 7, 'lost': 8, 'send': 9, 'info': 10, 'receiv': 11, 'packag': 12, 'order': 13, 'haven': 14, 'abl': 15, 'sent': 16, 'worri': 17, 'go': 18, 'deliv': 19, 'home': 20, 'issu': 21, 'status': 22, 'reciev': 23, 'date': 24, 'expect': 25, 'number': 26, 'came': 27, 'replac': 28, 'updat': 29, 'long': 30, 'ship': 31, 'credit': 32, 'mail': 33, 'check': 34, 'expedit': 35, 'period': 36, 'reason': 37, 'step': 38, 'get': 39, 'happen': 40, 'gotten': 41, 'longer': 42, 'time': 43, 'progress': 44, 'suppos': 45, 'shown': 46, 'courier': 47, 'day': 48, 'maximum': 49, 'request': 50, 'problem': 51, 'solut': 52, 'shouldn': 53, 'share': 54, 'rout': 55, 'possibl': 56, 'bank': 57, 'havent': 58, 'point': 59, 'thought': 60, 'avail': 61, 'appear': 62, 'put': 63, 'link': 64, 'activ': 65, 'continu': 66, 'dispos': 67, 'exist': 68, 'final': 69, 'physic': 70, 'jacket': 71, 'morn': 72, 'pocket': 73, 'reactiv': 74, 'previous': 75, 'r

# on filtre les valeurs extrèmes

In [68]:
# n filtre avec no_below=5, no_above=0.5 et keep_n=3000

dictionary.filter_extremes(
no_below=5, # garder les mots présents dans au moins 1 document
no_above=0.5, # supprimer les mots présents dans plus de 80% des documents
keep_n=5000 # garder au maximum 1000 mots
)
print('Vocabulaire final :', len(dictionary))

Vocabulaire final : 540


# On convertit en Bag Of Word pour avoir l'id + son nombre d'occurences

In [69]:
bow_corpus= [dictionary.doc2bow(doc) for doc in processed_docs]
for i, bow in enumerate(bow_corpus[:3]):
    print(df.loc[i, 'label'], '->', bow)

11 -> [(0, 1)]
11 -> [(1, 1), (2, 1), (3, 1)]
11 -> [(0, 1), (3, 1), (4, 1)]


# on entraîne le modèle LDA

In [70]:

# on crée une fonction réutilisable pour le model LDA, en faisant varier le nombre de topics directement en param
def training_LDA(nombre_topics):
    lda_model= gensim.models.LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=nombre_topics,
    random_state=42,
    passes= 15,
    iterations=100,
    alpha='auto',
    eta='auto'
    )
    lda_model.save("modele_lda.gensim")
    dictionary.save("dictionnaire_lda.gensim")
    
    for idx, topic in lda_model.print_topics(num_words=10):
        print(f'Topic{idx} -> {topic}')
    


In [71]:
# on teste tout d'abord avec 6 topics 

six_topics = training_LDA(6)


Topic0 -> 0.442*"countri" + 0.239*"get" + 0.020*"balanc" + 0.019*"day" + 0.017*"appl" + 0.017*"process" + 0.015*"coupl" + 0.013*"twice" + 0.013*"beneficiari" + 0.012*"intern"
Topic1 -> 0.199*"support" + 0.108*"cash" + 0.084*"exchang" + 0.080*"withdraw" + 0.053*"currenc" + 0.049*"rate" + 0.033*"verifi" + 0.025*"wrong" + 0.024*"fund" + 0.018*"accept"
Topic2 -> 0.140*"charg" + 0.110*"payment" + 0.051*"pend" + 0.043*"show" + 0.042*"refund" + 0.037*"receiv" + 0.036*"extra" + 0.028*"purchas" + 0.026*"statement" + 0.022*"fee"
Topic3 -> 0.057*"bank" + 0.045*"check" + 0.038*"ident" + 0.032*"hasn" + 0.030*"verif" + 0.027*"sent" + 0.027*"chequ" + 0.024*"phone" + 0.024*"deposit" + 0.021*"friend"
Topic4 -> 0.215*"transfer" + 0.053*"tri" + 0.048*"declin" + 0.040*"time" + 0.038*"transact" + 0.036*"work" + 0.032*"cancel" + 0.032*"go" + 0.020*"debit" + 0.018*"top"
Topic5 -> 0.248*"card" + 0.210*"avail" + 0.054*"long" + 0.040*"dispos" + 0.039*"chang" + 0.039*"virtual" + 0.037*"activ" + 0.029*"order" + 0

In [72]:
# on teste avec 8 topics

eight_topics = training_LDA(8)

Topic0 -> 0.332*"transfer" + 0.080*"receiv" + 0.043*"show" + 0.042*"declin" + 0.035*"long" + 0.032*"go" + 0.023*"mastercard" + 0.022*"day" + 0.020*"visa" + 0.019*"differ"
Topic1 -> 0.367*"countri" + 0.183*"support" + 0.183*"avail" + 0.031*"verifi" + 0.027*"fund" + 0.016*"accept" + 0.014*"ident" + 0.012*"coupl" + 0.012*"limit" + 0.010*"allow"
Topic2 -> 0.203*"charg" + 0.122*"exchang" + 0.076*"currenc" + 0.071*"rate" + 0.052*"extra" + 0.024*"abl" + 0.023*"fail" + 0.023*"cost" + 0.022*"wrong" + 0.021*"item"
Topic3 -> 0.181*"payment" + 0.083*"pend" + 0.054*"bank" + 0.044*"cancel" + 0.042*"check" + 0.030*"transact" + 0.030*"statement" + 0.023*"phone" + 0.020*"sent" + 0.020*"friend"
Topic4 -> 0.440*"card" + 0.079*"refund" + 0.036*"debit" + 0.031*"direct" + 0.028*"physic" + 0.027*"credit" + 0.022*"free" + 0.022*"lost" + 0.021*"request" + 0.018*"thought"
Topic5 -> 0.063*"long" + 0.056*"possibl" + 0.050*"fee" + 0.048*"deposit" + 0.041*"balanc" + 0.041*"wait" + 0.038*"verif" + 0.038*"expir" + 0.

In [73]:
# et enfin 10

ten_topics = training_LDA(10)

Topic0 -> 0.376*"transfer" + 0.088*"long" + 0.054*"cancel" + 0.048*"show" + 0.047*"bank" + 0.032*"pend" + 0.019*"declin" + 0.017*"china" + 0.017*"error" + 0.015*"morn"
Topic1 -> 0.394*"avail" + 0.105*"currenc" + 0.059*"fund" + 0.057*"exchang" + 0.032*"fail" + 0.026*"limit" + 0.023*"foreign" + 0.021*"allow" + 0.018*"right" + 0.017*"passcod"
Topic2 -> 0.135*"exchang" + 0.119*"rate" + 0.103*"refund" + 0.060*"wrong" + 0.052*"purchas" + 0.041*"top" + 0.040*"abl" + 0.038*"cost" + 0.037*"appl" + 0.036*"item"
Topic3 -> 0.338*"get" + 0.173*"payment" + 0.052*"verifi" + 0.051*"ident" + 0.035*"check" + 0.023*"problem" + 0.017*"reason" + 0.016*"gone" + 0.016*"servic" + 0.016*"verif"
Topic4 -> 0.559*"countri" + 0.279*"support" + 0.017*"explain" + 0.016*"free" + 0.014*"code" + 0.011*"doubl" + 0.010*"verif" + 0.009*"assist" + 0.007*"europ" + 0.005*"automat"
Topic5 -> 0.379*"charg" + 0.100*"chang" + 0.097*"extra" + 0.069*"statement" + 0.049*"wait" + 0.031*"complet" + 0.025*"recogn" + 0.019*"express" + 

In [74]:
lda_model= gensim.models.LdaModel(
corpus=bow_corpus,
id2word=dictionary,
num_topics=8,
random_state=42,
passes= 15,
iterations=100,
alpha='auto',
eta='auto'
)
lda_model.save("modele_lda.gensim")
dictionary.save("dictionnaire_lda.gensim")

In [ ]:
for i, bow in enumerate(bow_corpus):
    distrib= lda_model.get_document_topics(bow)
    print(df.loc[i, 'label'], '->', distrib)

# visualisation avec pyLDAvis

In [76]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

vis = gensimvis.prepare(lda_model, bow_corpus, dictionary)
pyLDAvis.display(vis)